# Notebook 02 — Feature Engineering

**Project:** Fraud Detection & Strategy Analytics  
**Objective:** Transform raw transaction data into a rich, model-ready feature matrix by engineering time-based signals, amount patterns, risk aggregations, and encoded categoricals.

---

## Outline
1. Load & prepare raw data
2. Time-based features
3. Amount-based features
4. Risk aggregation features
5. Categorical encoding
6. Feature correlation with fraud label
7. Save processed feature matrix

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.generate_data import generate_dataset
from src.features import (
    add_time_features,
    add_amount_features,
    add_risk_features,
    encode_categoricals,
    build_feature_matrix,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
print('Ready.')

## 1. Load Data

In [ ]:
df_raw = generate_dataset(n=100_000)
print(f'Raw shape: {df_raw.shape}')
print(f'Fraud rate: {df_raw["is_fraud"].mean():.2%}')
df_raw.head(3)

## 2. Time-Based Features

Raw temporal columns (`time_of_day`, `day_of_week`) encode useful signals but miss nuances:
- **Cyclic encoding** ensures the model understands that 23:00 and 00:00 are adjacent.
- **is_night** and **is_weekend** create interpretable binary risk flags.
- **is_business_hours** captures the complement — transactions when banks are staffed.

In [ ]:
df_time = add_time_features(df_raw)

new_cols = ['is_weekend', 'is_night', 'is_business_hours', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
print('New time features:')
print(df_time[new_cols].describe().round(3))

In [ ]:
# Validate: is_night fraud rate should be higher
night_rates = df_time.groupby('is_night')['is_fraud'].mean()
print('Fraud rate — daytime vs night:')
print(night_rates.rename({0: 'Daytime', 1: 'Night-time'}).to_string())

## 3. Amount-Based Features

- `log_amount` reduces the right-skew of transaction amounts for linear models.
- `amount_bucket` captures non-linear risk tiers without requiring the model to learn them.
- `is_round_amount` flags card-testing behaviour where fraudsters probe cards with exact-dollar transactions.
- `amount_velocity_ratio` measures spend intensity — a large purchase combined with high velocity is highly suspicious.

In [ ]:
df_amt = add_amount_features(df_time)

amt_cols = ['log_amount', 'amount_bucket', 'is_round_amount', 'amount_velocity_ratio']
print('New amount features:')
print(df_amt[amt_cols].describe().round(3))

In [ ]:
# Validate: log_amount comparison by class
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color, name in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    subset = df_amt[df_amt['is_fraud'] == label]
    axes[0].hist(subset['log_amount'], bins=50, alpha=0.6, color=color, label=name, density=True)
    axes[1].hist(subset['amount_velocity_ratio'].clip(0, 500), bins=50, alpha=0.6, color=color, label=name, density=True)

axes[0].set_title('Log(Amount): Fraud vs Legitimate')
axes[0].set_xlabel('log(1 + amount)')
axes[0].legend()

axes[1].set_title('Amount / Velocity Ratio')
axes[1].set_xlabel('transaction_amount / (velocity + 1)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Risk Aggregation Features

Composite risk signals combine multiple individual indicators:
- `high_risk_merchant`: Binary flag for travel / electronics MCCs.
- `tenure_risk`: New accounts (<90 days) have limited fraud history and are targeted by fraudsters.
- `age_velocity_risk`: Young customers with burst activity — a pattern seen in account takeovers.
- `risk_signal_count`: Additive composite — the more flags, the higher the risk.

In [ ]:
df_risk = add_risk_features(df_amt)

risk_cols = ['high_risk_merchant', 'tenure_risk', 'age_velocity_risk', 'risk_signal_count']

# Fraud rate by risk signal count
rsc_fraud = df_risk.groupby('risk_signal_count')['is_fraud'].agg(['mean', 'count'])
rsc_fraud.columns = ['fraud_rate', 'count']
rsc_fraud['fraud_rate_pct'] = rsc_fraud['fraud_rate'] * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(rsc_fraud.index, rsc_fraud['fraud_rate_pct'], color=sns.color_palette('Reds', len(rsc_fraud)))
ax.set_xlabel('Number of Risk Signals Present')
ax.set_ylabel('Fraud Rate (%)')
ax.set_title('Fraud Rate vs. Composite Risk Signal Count')
for x, y in zip(rsc_fraud.index, rsc_fraud['fraud_rate_pct']):
    ax.text(x, y + 0.2, f'{y:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(rsc_fraud.to_string())

> **Insight:** Each additional risk signal roughly doubles the fraud rate. Transactions with 3+ signals are extremely high risk and should almost always be declined.

## 5. Categorical Encoding

Categorical features (`merchant_category`, `device_type`) are converted to binary dummies.
We drop the first level (`drop_first=True`) to avoid perfect multicollinearity in linear models.

In [ ]:
df_encoded = encode_categoricals(df_risk)

# Show columns added by encoding
encoded_cols = [c for c in df_encoded.columns
                if c.startswith('merchant_category_') or c.startswith('device_type_')]
print(f'Encoded columns ({len(encoded_cols)}): {encoded_cols}')
print(f'\nFull dataset shape after encoding: {df_encoded.shape}')

## 6. Full Feature Matrix

Apply the full pipeline via `build_feature_matrix()`.

In [ ]:
df_features = build_feature_matrix(df_raw)

# Identify feature columns (exclude ID, timestamp, target)
exclude = ['transaction_id', 'timestamp', 'is_fraud', 'amount_bucket_label', 'label']
feature_cols = [c for c in df_features.columns if c not in exclude]

print(f'Total engineered features: {len(feature_cols)}')
print('\nFeature list:')
for c in feature_cols:
    print(f'  {c}')

## 7. Feature Importance Preview — Correlation with Fraud Label

In [ ]:
# Point-biserial correlation (equivalent to Pearson for binary target)
feature_corr = (
    df_features[feature_cols + ['is_fraud']]
    .corr()['is_fraud']
    .drop('is_fraud')
    .abs()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 7))
feature_corr.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('|Correlation with is_fraud|')
ax.set_title('Top 20 Features by Absolute Correlation with Fraud Label')
plt.tight_layout()
plt.show()

## 8. Save Processed Features

Persist the engineered feature matrix for use in model training notebooks.

In [ ]:
from pathlib import Path

save_path = Path('../data/features.csv')

save_cols = feature_cols + ['is_fraud', 'transaction_id']
save_cols = [c for c in save_cols if c in df_features.columns]

df_features[save_cols].to_csv(save_path, index=False)
print(f'Feature matrix saved to {save_path}')
print(f'Shape: {df_features[save_cols].shape}')

## Summary

| Category | New Features | Key Insight |
|----------|-------------|-------------|
| Time | is_night, is_weekend, hour_sin/cos, dow_sin/cos, is_business_hours | Cyclic encoding; off-hours = higher risk |
| Amount | log_amount, amount_bucket, is_round_amount, amount_velocity_ratio | Log transform reduces skew; round amounts flag card-testing |
| Risk | high_risk_merchant, tenure_risk, age_velocity_risk, risk_signal_count | Composite signals amplify predictive power |
| Encoding | merchant_category dummies, device_type dummies | One-hot with drop_first avoids collinearity |

These features will be consumed directly by the models in Notebook 03.